In [1]:
import pybigtools

bw = pybigtools.open(
    "../data/raw/liver_h3k27ac_foldchange.bigWig"
)

print(bw.chroms())

{'chr1': 248956422, 'chr10': 133797422, 'chr11': 135086622, 'chr11_KI270721v1_random': 100316, 'chr12': 133275309, 'chr13': 114364328, 'chr14': 107043718, 'chr14_GL000009v2_random': 201709, 'chr14_GL000194v1_random': 191469, 'chr14_GL000225v1_random': 211173, 'chr14_KI270722v1_random': 194050, 'chr14_KI270723v1_random': 38115, 'chr14_KI270724v1_random': 39555, 'chr14_KI270725v1_random': 172810, 'chr15': 101991189, 'chr15_KI270727v1_random': 448248, 'chr16': 90338345, 'chr16_KI270728v1_random': 1872759, 'chr17': 83257441, 'chr17_GL000205v2_random': 185591, 'chr17_KI270729v1_random': 280839, 'chr17_KI270730v1_random': 112551, 'chr18': 80373285, 'chr19': 58617616, 'chr1_KI270706v1_random': 175055, 'chr1_KI270707v1_random': 32032, 'chr1_KI270708v1_random': 127682, 'chr1_KI270709v1_random': 66860, 'chr1_KI270710v1_random': 40176, 'chr1_KI270711v1_random': 42210, 'chr1_KI270712v1_random': 176043, 'chr1_KI270713v1_random': 40745, 'chr1_KI270714v1_random': 41717, 'chr2': 242193529, 'chr20': 64

In [2]:
values = bw.values(
    "chr1",
    100000,
    100100
)

print(values[:10])

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [3]:
import numpy as np

signal = np.array(values)

mean_signal = np.nanmean(values)

print(mean_signal)

0.0


In [5]:
from pybigtools import open

bb = open(
    "../data/raw/ENCFF106WBX.bigBed"
)

In [4]:
WINDOW_SIZE = 200

def extract_fixed_window(
    genome,
    chrom,
    start,
    peak_offset,
    window_size=WINDOW_SIZE
):

    summit = start + peak_offset

    half_window = window_size // 2

    window_start = summit - half_window
    window_end = summit + half_window

    sequence = genome[
        chrom
    ][
        window_start:window_end
    ].seq.upper()

    return sequence

In [7]:
from pyfaidx import Fasta
genome = Fasta(
    "../data/reference/hg38.fa"
)

In [8]:
positive_sequences = []

chromosomes = [
    "chr1",
    "chr2",
    "chr3"
]

for chrom in chromosomes:

    print(f"Processing {chrom}")

    chrom_size = bb.chroms()[chrom]

    records = list(
        bb.records(
            chrom,
            0,
            chrom_size
        )
    )

    for record in records:

        (
            start,
            end,
            name,
            score,
            strand,
            signalValue,
            pValue,
            qValue,
            peak
        ) = record

        sequence = extract_fixed_window(
            genome=genome,
            chrom=chrom,
            start=start,
            peak_offset=int(peak)
        )

        if len(sequence) != 200:
            continue

        positive_sequences.append({
            "chrom": chrom,
            "sequence": sequence,
            "label": 1
        })

Processing chr1
Processing chr2
Processing chr3


In [9]:
positive_sequences

[{'chrom': 'chr1',
  'sequence': 'CCGCCCTTGTGACGTCACGGAAGGCGCACCCTTGTGACGTCACAGGGGACTACCACTCACGCAGAGCCAATCAGAACTCGCGGTGGGGGCTGCTGGTTCTTCCAGGAGCGCGCATGAGCGGACGCTGCCTACTGGTGGCCGGGCGGGATGTAACCGGCTGCTGAGCTGGCAGTTCTGTGTCGCTAGGCTTCTGCCCGGCC',
  'label': 1},
 {'chrom': 'chr1',
  'sequence': 'CAGAAGCTCCTCAATGGCCAGCGCCAGCTGCAGCCCCGGCCGCCCACTCGCCTCATCTGAGCCTGGGTACGTGCGCTCCACAACGCCTCCCCCAGCCAGGGCCCGGGGATCCCCGGGAGCGTCCCCGGCTACCTGGCGCCGCTCATCCTGGGTAGGGTCGGCCCCCTGAGGCTGCCCGGCATGAGGGAGTTGCACCCCTG',
  'label': 1},
 {'chrom': 'chr1',
  'sequence': 'GAAAAGGGAAGAATGCAAAAGTCAAAGACACGTCACCCTCCTTGAGACAGCCCTCCAGTCCAGGCCAAATCTCAGCCTGCCCTTGGTCCGCTGTGGTTGGGCCTGCACCCAAGCCATGAGCACACGCAGCAATTGTGGCAGCAGAAGCTTCCTCTGGGCTCAGACTCAGGCTGATGCTGCGTCAGGACCTGCCGCGGTCT',
  'label': 1},
 {'chrom': 'chr1',
  'sequence': 'GGACCTTGGCTCCGGATAATCCGTTTCCGGGTCAACAAAAAACGTCGCGCGAGGGGCGGGGCGCGTACGTGCAGGGAGGGGAGGCAGAGAAAAAGGCGGGGCCGGGCCGGGCCGGGGCGGGGTCTCGGGCAGGGGCGGGGAGCTTACCGACCTCCCGCCCCCGCTGCGCGCGTTTCTGGCCCTGCCAGTGTCTCCGCCGG',
  'labe

In [10]:
h3k27ac_signals = []

In [20]:
import pandas as pd

dataset_df = pd.read_csv(
    "../data/processed/liver_accessibility_gc_matched_coords.csv"
)

In [21]:
dataset_df.keys()

Index(['chrom', 'start', 'end', 'sequence', 'label'], dtype='object')

In [22]:
for idx, row in dataset_df.iterrows():

    chrom = row["chrom"]

    start = row["start"]

    end = row["end"]

    values = bw.values(
        chrom,
        start,
        end
    )

    mean_signal = np.nanmean(values)

    if np.isnan(mean_signal):

        mean_signal = 0.0

    h3k27ac_signals.append(
        mean_signal
    )

    if idx % 1000 == 0:

        print(idx)

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000


In [23]:
dataset_df["h3k27ac"] = h3k27ac_signals

In [24]:
dataset_df["h3k27ac"].describe()

count    49416.000000
mean         6.672992
std          7.479480
min          0.000000
25%          0.000000
50%          3.308830
75%         15.580666
max         71.265356
Name: h3k27ac, dtype: float64

In [25]:
positive_signals = dataset_df[
    dataset_df["label"] == 1
]["h3k27ac"]

negative_signals = dataset_df[
    dataset_df["label"] == 0
]["h3k27ac"]

print(positive_signals.mean())

print(negative_signals.mean())

7.403286138383175
5.9426985268379005


In [29]:
dataset_df.to_csv(
    "../data/processed/liver_accessibility_multimodal.csv",
    index=False
)

In [27]:
dataset_df["h3k27ac"] = np.log1p(
    dataset_df["h3k27ac"]
)

In [28]:
dataset_df["h3k27ac"].describe()

count    49416.000000
mean         1.408986
std          1.208910
min          0.000000
25%          0.000000
50%          1.460666
75%          2.808237
max          4.280345
Name: h3k27ac, dtype: float64